In [12]:
import os
import pandas as pd
from sqlalchemy import create_engine

In [13]:
# Database Connection Setup

DB_USER = 'root'
DB_PASSWORD = 'Enter your Password'
DB_HOST = 'localhost'
DB_PORT = '3306'
DB_NAME = 'olist_analytics'

# Create SQLAlchemy connection engine
engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

print("Starting ETL pipeline...")

Starting ETL pipeline...


In [14]:
# 1. Ingest & Clean Product Categories (Translation Mapping)

df_products = pd.read_csv('data/olist_products_dataset.csv')
df_trans = pd.read_csv('data/product_category_name_translation.csv')

# Merge translation and impute missing categories
df_products = df_products.merge(df_trans, on='product_category_name', how='left')
df_products['product_category_name_english'] = df_products['product_category_name_english'].fillna('other')
df_products = df_products.drop(columns=['product_category_name'])
df_products.to_sql('dim_products', engine, if_exists='replace', index=False)
print("✔ 1/6: dim_products loaded.")

✔ 1/6: dim_products loaded.


In [15]:
# 2. Ingest & Parse Orders (Date Conversions & Status Filtering)

df_orders = pd.read_csv('data/olist_orders_dataset.csv')
date_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_columns:
    df_orders[col] = pd.to_datetime(df_orders[col])

# Filter for delivered orders to analyze real SLA fulfillment metrics
df_orders_delivered = df_orders[df_orders['order_status'] == 'delivered'].copy()
df_orders_delivered.to_sql('fact_orders', engine, if_exists='replace', index=False)
print("✔ 2/6: fact_orders loaded.")

✔ 2/6: fact_orders loaded.


In [16]:
# 3. Ingest Customers Dimension Table

df_customers = pd.read_csv('data/olist_customers_dataset.csv')
df_customers['customer_city'] = df_customers['customer_city'].str.strip().str.title()
df_customers['customer_state'] = df_customers['customer_state'].str.strip().str.upper()
df_customers.to_sql('dim_customers', engine, if_exists='replace', index=False)
print("✔ 3/6: dim_customers loaded.")

✔ 3/6: dim_customers loaded.


In [17]:
# 4. Ingest Sellers Dimension Table

df_sellers = pd.read_csv('data/olist_sellers_dataset.csv')
df_sellers['seller_city'] = df_sellers['seller_city'].str.strip().str.title()
df_sellers['seller_state'] = df_sellers['seller_state'].str.strip().str.upper()
df_sellers.to_sql('dim_sellers', engine, if_exists='replace', index=False)
print("✔ 4/6: dim_sellers loaded.")

✔ 4/6: dim_sellers loaded.


In [18]:
# 5. Ingest Order Items Fact Table

df_items = pd.read_csv('data/olist_order_items_dataset.csv')
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'])
df_items.to_sql('fact_order_items', engine, if_exists='replace', index=False)
print("✔ 5/6: fact_order_items loaded.")

✔ 5/6: fact_order_items loaded.


In [19]:
# 6. Ingest Payments Fact Table

df_payments = pd.read_csv('data/olist_order_payments_dataset.csv')
df_payments.to_sql('fact_payments', engine, if_exists='replace', index=False)
print("✔ 6/6: fact_payments loaded.")

print("\nETL Execution Complete: Relational Star-Schema loaded into MySQL 'olist_analytics'!")

✔ 6/6: fact_payments loaded.

ETL Execution Complete: Relational Star-Schema loaded into MySQL 'olist_analytics'!
